# Stage 1 — Fine-tuning CamemBERT sur Distant Supervision

**Objectif** : entraîner `camembert-base` sur les 6936 docs annotés automatiquement (distant supervision 1973/1978) pour que le modèle apprenne le domaine électoral français.

**Résultat attendu** : un modèle de domaine sauvegardé dans `/content/stage1_model/`, qui sera utilisé en Stage 2.

| Dataset | Docs | Source |
|---------|-----:|--------|
| train | 6936 | distant supervision 1973/1978 |
| val | 867 | distant supervision 1973/1978 |
| test | 867 | distant supervision 1973/1978 |

In [1]:
# ── Installation ──────────────────────────────────────────────────────────────
!pip install transformers datasets accelerate seaborn -q

import os, json, shutil
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict

print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
operator torchvision::nms does not exist

In [2]:
# ── Montage Google Drive et chargement des donnees ───────────────────────────
# Assure-toi d avoir uploade les 4 fichiers dans Drive : archelec_ner/bio_distantsup/

from google.colab import drive
drive.mount("/content/drive")

DRIVE_FOLDER = "/content/drive/MyDrive/archelec_ner/bio_distantsup/"

fichiers = ["train.json", "val.json", "test.json", "label2id.json"]
for f in fichiers:
    src = DRIVE_FOLDER + f
    if os.path.exists(src):
        shutil.copy(src, f"/content/{f}")
        taille = os.path.getsize(f"/content/{f}") / 1024 / 1024
        print(f"OK {f} ({taille:.1f} MB)")
    else:
        print(f"MANQUANT {f} -- verifie le chemin Drive : {src}")

Selectionne les 4 fichiers de bio_distantsup/


KeyboardInterrupt: 

In [3]:
# ── Chargement label mapping ──────────────────────────────────────────────────
with open('/content/label2id.json') as f:
    LABEL2ID = json.load(f)
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
print(f'Labels ({NUM_LABELS}) : {LABEL2ID}')

# ── Chargement des splits ─────────────────────────────────────────────────────
def charger_split(chemin):
    with open(chemin, encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_list([
        {
            'input_ids': d['input_ids'],
            'ner_tags':  d['ner_tags'],
            'id':        d['id'],
            'annee':     d['annee']
        }
        for d in data
    ])

dataset = DatasetDict({
    'train': charger_split('/content/train.json'),
    'val':   charger_split('/content/val.json'),
    'test':  charger_split('/content/test.json'),
})

print(f'Train : {len(dataset["train"])} docs')
print(f'Val   : {len(dataset["val"])} docs')
print(f'Test  : {len(dataset["test"])} docs')

# Distribution des labels dans le train
tag_counts = defaultdict(int)
for ex in dataset['train']:
    for tag in ex['ner_tags']:
        tag_counts[ID2LABEL[tag]] += 1
print('\nDistribution labels train :')
for tag, count in sorted(tag_counts.items()):
    print(f'  {tag:<10} : {count:>8}')

FileNotFoundError: [Errno 2] No such file or directory: '/content/label2id.json'

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
MAX_LENGTH   = 512
PAD_TOKEN_ID = 1  # <pad> CamemBERT

def preprocess_batch(examples):
    batch_input_ids      = []
    batch_attention_mask = []
    batch_labels         = []

    for input_ids, ner_tags in zip(examples['input_ids'], examples['ner_tags']):
        input_ids = input_ids[:MAX_LENGTH]
        ner_tags  = ner_tags[:MAX_LENGTH]
        pad_len        = MAX_LENGTH - len(input_ids)
        attention_mask = [1] * len(input_ids) + [0] * pad_len
        input_ids      = input_ids + [PAD_TOKEN_ID] * pad_len
        labels         = ner_tags  + [-100] * pad_len
        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(labels)

    return {
        'input_ids':      batch_input_ids,
        'attention_mask': batch_attention_mask,
        'labels':         batch_labels,
    }

dataset_proc = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=['ner_tags', 'id', 'annee']
)
dataset_proc.set_format('torch')
print('Dataset pret pour le Trainer')
print(f'Features : {dataset_proc["train"].features}')

In [ ]:
# ── Fonction de métriques ─────────────────────────────────────────────────────
def extraire_spans(tags):
    spans = set()
    i = 0
    while i < len(tags):
        tag = tags[i] if isinstance(tags[i], str) else ID2LABEL[tags[i]]
        if tag.startswith('B-'):
            entite = tag[2:]
            debut  = i
            i += 1
            while i < len(tags):
                t = tags[i] if isinstance(tags[i], str) else ID2LABEL[tags[i]]
                if t == f'I-{entite}':
                    i += 1
                else:
                    break
            spans.add((entite, debut, i))
        else:
            i += 1
    return spans

def compute_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)
    stats        = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    stats_global = {'tp': 0, 'fp': 0, 'fn': 0}

    for pred_seq, label_seq in zip(predictions, labels):
        pred_clean  = [ID2LABEL[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        label_clean = [ID2LABEL[l] for l in label_seq if l != -100]
        spans_pred  = extraire_spans(pred_clean)
        spans_gold  = extraire_spans(label_clean)

        for span in spans_pred:
            if span in spans_gold:
                stats[span[0]]['tp'] += 1
                stats_global['tp']   += 1
            else:
                stats[span[0]]['fp'] += 1
                stats_global['fp']   += 1
        for span in spans_gold:
            if span not in spans_pred:
                stats[span[0]]['fn'] += 1
                stats_global['fn']   += 1

    def prf(tp, fp, fn):
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2*p*r / (p+r)  if (p  + r) > 0 else 0
        return p*100, r*100, f1*100

    res = {}
    for entite in ['PER', 'ORG', 'LOC', 'MISC']:
        s = stats[entite]
        p, r, f1 = prf(s['tp'], s['fp'], s['fn'])
        res[f'f1_{entite}']        = round(f1, 2)
        res[f'precision_{entite}'] = round(p,  2)
        res[f'recall_{entite}']    = round(r,  2)
    p_g, r_g, f1_g = prf(stats_global['tp'], stats_global['fp'], stats_global['fn'])
    res['f1_global']        = round(f1_g, 2)
    res['precision_global'] = round(p_g,  2)
    res['recall_global']    = round(r_g,  2)
    return res

print('compute_metrics defini')

In [ ]:
# ── Entraînement Stage 1 — CamemBERT-base sur distant supervision ─────────────
MODEL_NAME = 'camembert-base'
print(f'Chargement {MODEL_NAME}...')

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

args = TrainingArguments(
    output_dir                  = '/content/stage1_checkpoints',
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_global',
    greater_is_better           = True,
    logging_steps               = 200,
    fp16                        = True,
    report_to                   = 'none',
    save_total_limit            = 1,
)

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = dataset_proc['train'],
    eval_dataset    = dataset_proc['val'],
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Entrainement Stage 1...')
trainer.train()
print('Entrainement termine !')

In [ ]:
# ── Évaluation sur le test set (distant supervision) ─────────────────────────
print('=== EVALUATION STAGE 1 sur TEST (distant supervision) ===')
results = trainer.evaluate(dataset_proc['test'])

print()
print('{:<10} {:>12} {:>10} {:>10}'.format('Entite', 'Precision', 'Recall', 'F1'))
print('-' * 45)
for entite in ['PER', 'ORG', 'LOC', 'MISC']:
    p  = results.get('eval_precision_' + entite, 0)
    r  = results.get('eval_recall_'    + entite, 0)
    f1 = results.get('eval_f1_'        + entite, 0)
    print('{:<10} {:>11.2f}% {:>9.2f}% {:>9.2f}%'.format(entite, p, r, f1))
print('-' * 45)
p_g  = results.get('eval_precision_global', 0)
r_g  = results.get('eval_recall_global',    0)
f1_g = results.get('eval_f1_global',        0)
print('{:<10} {:>11.2f}% {:>9.2f}% {:>9.2f}%'.format('GLOBAL', p_g, r_g, f1_g))

# Sauvegarder les résultats
with open('/content/stage1_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResultats sauvegardes dans /content/stage1_results.json')

In [ ]:
# ── Sauvegarder le modèle Stage 1 ────────────────────────────────────────────
# Ce modèle sera rechargé en Stage 2 pour le fine-tuning sur annotations manuelles

trainer.save_model('/content/stage1_model')
print('Modele Stage 1 sauvegarde dans /content/stage1_model')
print()
print('Fichiers du modele :')
for f in os.listdir('/content/stage1_model'):
    taille = os.path.getsize(f'/content/stage1_model/{f}') / 1024 / 1024
    print(f'  {f} ({taille:.1f} MB)')
print()
print('ETAPE SUIVANTE : telecharger /content/stage1_model/ et lancer stage2_manual.ipynb')